In [ ]:
import sys
!{sys.executable} -m pip install indic-nlp-library
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
from collections import defaultdict
from indicnlp.tokenize import sentence_tokenize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.8 MB/s eta 0:00:00


In [ ]:
wordnet_senses = [
"डोळा","मार्ग","वाट","वाटणे","देणे","भाव","योग","अर्थ","नाव",
"वर","बरोबर","हलका","मंद","फोडणे","उडणे","वार","गंध","पाठ","अंक","गुण"
]

In [ ]:
homepages = [
"https://www.loksatta.com/",
"https://www.esakal.com/",
"https://www.lokmat.com/",
"https://marathi.abplive.com/",
"https://www.bbc.com/marathi",
"https://www.tv9marathi.com/",
"https://news18marathi.com/",
"https://maharashtratimes.com/",
"https://mitraho.wordpress.com/",
"https://marathispandan.com/",
"https://maayboli.com/"
]

In [ ]:
def extract_article_links(home_url, limit=100):

    links=set()

    try:
        r=requests.get(home_url,timeout=10)
        soup=BeautifulSoup(r.text,"html.parser")

        for a in soup.find_all("a",href=True):

            href=a["href"]
            full=urljoin(home_url,href)

            if any(x in full for x in ["news","article","story","202"]):
                links.add(full)

            if len(links)>=limit:
                break

    except:
        pass

    return list(links)

In [ ]:
article_urls=[]

for home in homepages:

    urls=extract_article_links(home)
    article_urls.extend(urls)

article_urls=list(set(article_urls))

print("Total articles:",len(article_urls))

Total articles: 523


In [ ]:
def scrape_article(url):

    try:
        r=requests.get(url,timeout=10)
        soup=BeautifulSoup(r.text,"html.parser")

        paragraphs=soup.find_all("p")

        text=" ".join(p.get_text() for p in paragraphs)

        return text

    except:
        return ""

In [ ]:
raw_texts=[]

for url in article_urls:

    text=scrape_article(url)

    if text and len(text)>300:
        raw_texts.append(text)

print("Articles collected:",len(raw_texts))

Articles collected: 389


In [ ]:
def clean_text(text):

    text=re.sub(r"http\S+","",text)
    text=re.sub(r"\n+"," ",text)

    text=re.sub(r"[^\u0900-\u097F0-9\s.,?!।]"," ",text)

    text=re.sub(r"\s+"," ",text)

    return text.strip()

In [ ]:
cleaned_texts=[clean_text(t) for t in raw_texts]

In [ ]:
all_sentences=[]

for text in cleaned_texts:

    sents=sentence_tokenize.sentence_split(text,lang="mr")

    for s in sents:

        s=s.strip()

        if len(s)>25:
            all_sentences.append(s)

all_sentences=list(set(all_sentences))

print("Total sentences:",len(all_sentences))

Total sentences: 11430


In [ ]:
output_file = "raw_marathi_sentences.txt"

with open(output_file, "w", encoding="utf-8") as f:

    for sentence in all_sentences:
        f.write(sentence + "\n")

print("File saved:", output_file)

File saved: raw_marathi_sentences.txt


In [ ]:
from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>